# SemiMind AI – Phase 1: Dataset Inspection Overview

> **Phase 1 | Notebook 1 of 3**  
> Purpose: Project introduction, dataset descriptions, and initial cross-dataset inspection.

---

## Project Introduction

**SemiMind AI** is an AI-driven semiconductor manufacturing intelligence platform.  
It processes industrial and semiconductor manufacturing data to:

- Identify abnormal process patterns
- Predict equipment failures and quality issues
- Monitor data distributions over time
- Provide AI-powered decision support for process engineers

---

## Phase 1 Objective

Establish a clean, reproducible, well-documented foundation including:

1. Dataset inspection and validation
2. Automated profiling and report generation
3. Configuration-driven loading infrastructure
4. Documented data quality findings

---

## Dataset Descriptions

| Dataset | Source | Rows | Columns | Target |
|---------|--------|------|---------|--------|
| AI4I 2020 Predictive Maintenance | UCI ML Repo #601 | 10,000 | 14 | Machine failure (0/1) |
| SECOM Semiconductor Manufacturing | UCI ML Repo #179 | 1,567 | 590 features | Pass/Fail (-1/1) |


In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

In [ ]:
import pandas as pd
import numpy as np

from src.utils.config import load_config
from src.utils.logger import setup_logging
from src.ingestion.dataset_loader import load_ai4i_from_config, load_secom_from_config

# Initialise logging
setup_logging(level='WARNING', log_to_file=False)

# Load configuration
cfg = load_config()
print('Configuration loaded successfully.')
print(f'  Raw data root  : {cfg.raw_data_root}')
print(f'  Reports root   : {cfg.reports_root}')

## Dataset Loading

In [ ]:
# Load AI4I
ai4i_df = load_ai4i_from_config(cfg)
print(f'AI4I loaded → shape: {ai4i_df.shape}')

# Load SECOM
secom_features, secom_labels = load_secom_from_config(cfg)
print(f'SECOM features loaded → shape: {secom_features.shape}')
print(f'SECOM labels loaded   → shape: {secom_labels.shape}')

## Dataset Dimensions Comparison

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['AI4I 2020 Predictive Maintenance', 'SECOM Semiconductor Manufacturing'],
    'Rows': [ai4i_df.shape[0], secom_features.shape[0]],
    'Features / Columns': [ai4i_df.shape[1], secom_features.shape[1]],
    'Target Column': ['Machine failure', 'label (separate file)'],
    'Target Values': ['0 (no fail) / 1 (fail)', '-1 (pass) / 1 (fail)'],
    'Has Missing Values': [
        ai4i_df.isnull().any().any(),
        secom_features.isnull().any().any()
    ]
})
summary

## Column Inspection

In [ ]:
print('=== AI4I Columns ===')
for col in ai4i_df.columns:
    print(f'  {col:35s} dtype: {str(ai4i_df[col].dtype):10s}  missing: {ai4i_df[col].isnull().sum()}')

In [ ]:
print('=== SECOM Feature Columns (first 10 and last 5) ===')
cols = secom_features.columns.tolist()
for col in cols[:10]:
    print(f'  {col:15s} dtype: {str(secom_features[col].dtype):10s}  missing: {secom_features[col].isnull().sum()}')
print('  ...')
for col in cols[-5:]:
    print(f'  {col:15s} dtype: {str(secom_features[col].dtype):10s}  missing: {secom_features[col].isnull().sum()}')

print(f'\n=== SECOM Label Columns ===')
for col in secom_labels.columns:
    print(f'  {col:15s} dtype: {str(secom_labels[col].dtype):10s}')

## Initial Data Quality Checks

In [ ]:
from src.validation.dataset_validator import DatasetValidator

validator = DatasetValidator()

ai4i_report = validator.validate_ai4i(ai4i_df)
secom_report = validator.validate_secom(secom_features, secom_labels)

print(ai4i_report.summary())
print()
print(secom_report.summary())

---
**End of Notebook 1.** Continue with `02_ai4i_initial_analysis.ipynb` and `03_secom_initial_analysis.ipynb` for dataset-specific analysis.